# Otimização direcionada de RUL — FD001 — versão 2

Nesta etapa, será feita uma segunda busca de hiperparâmetros somente para os modelos que apresentaram sinais de pressão nos limites da busca anterior. A primeira busca será preservada como referência, e os estudos desta etapa serão gravados com a identificação `v2`.

O Optuna será utilizado para testar combinações de configurações de forma organizada, mantendo a validação por motor. Essa busca não transforma automaticamente um modelo em uma solução definitiva; ela apenas procura uma configuração mais adequada dentro das alternativas permitidas (Akiba et al., 2019).

A comparação continuará sendo feita pelo erro médio absoluto calculado primeiro dentro de cada motor e depois resumido entre os motores. Assim, uma trajetória longa não dominará o resultado apenas por possuir mais observações (Pedregosa et al., 2011).

In [1]:
from pathlib import Path
import sys
import optuna
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'vscode'
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import add_rul_label, candidate_rul_features, nonconstant_features
from cmapss_optuna import FINALIST_NAMES, optimize_model, study_summary
from multilang import LangMap

lang = LangMap(source='pt', target='pt')
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Preparação dos dados

Será utilizada a mesma base FD001 e o mesmo conjunto de atributos da versão anterior. Essa repetição é importante porque permite que a diferença observada seja atribuída principalmente à ampliação seletiva dos hiperparâmetros, e não a uma mudança escondida nos dados ou na validação.

In [2]:
fd001 = add_rul_label(load_split(DATA_DIR, 'train', 'FD001'))
features = nonconstant_features(
    fd001,
    candidate_rul_features(include_settings=True),
)
features

['cycle',
 'setting1',
 'setting2',
 's2',
 's3',
 's4',
 's6',
 's7',
 's8',
 's9',
 's11',
 's12',
 's13',
 's14',
 's15',
 's17',
 's20',
 's21']

## 2. Busca v2 direcionada

Na versão 2, os limites do `HistGradientBoosting` serão ampliados para permitir árvores menores e folhas com mais observações. Também será permitido um número maior de iterações. No `XGBoost`, será ampliado o limite de `gamma`, que controla a exigência mínima para uma nova divisão, e o número máximo de árvores. CatBoost e Extra Trees permanecerão na versão 1 porque não apresentaram pressão suficiente nos limites observados.

A busca continuará usando cinco grupos de motores e o mesmo critério de erro. Os bancos da versão 1 não serão sobrescritos.

In [3]:
v2_models = ('HistGradientBoosting', 'XGBoost')
studies_v2 = {}
for model_name in v2_models:
    studies_v2[model_name] = optimize_model(
        fd001,
        features=features,
        model_name=model_name,
        n_trials=100,
        n_splits=5,
        random_state=42,
        storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
        search_version='v2',
    )
summary_v2 = study_summary(studies_v2)
summary_v2[['modelo', 'versao_busca', 'mae_cv_motor', 'trials', 'melhor_trial', 'status', 'hiperparametros']].round(2)

,modelo,versao_busca,mae_cv_motor,trials,melhor_trial,status,hiperparametros
0,XGBoost,v2,25.32,100,76,COMPLETE,feature_selection=passthrough | n_estimators=1...
1,HistGradientBoosting,v2,25.43,100,14,COMPLETE,feature_selection=select_k_best | k_features=1...


### Insights
A tabela acima deve ser lida como uma verificação de refinamento. Caso o erro da versão 2 permaneça muito próximo do erro da versão 1, será indicado que o ganho possível por ajuste de hiperparâmetros já é pequeno. Nesse cenário, a próxima fonte de melhoria provavelmente estará na representação temporal dos dados, como o uso do valor recente, da inclinação e de janelas de observações, e não em uma busca ainda mais ampla dos mesmos controles.

In [4]:
studies_v1 = {}
for model_name in FINALIST_NAMES:
    studies_v1[model_name] = optimize_model(
        fd001,
        features=features,
        model_name=model_name,
        n_trials=100,
        n_splits=5,
        random_state=42,
        storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
        search_version='v1',
    )
comparison = pd.concat([study_summary(studies_v1), summary_v2], ignore_index=True)
comparison = comparison.sort_values('mae_cv_motor').reset_index(drop=True)
comparison[['modelo', 'versao_busca', 'mae_cv_motor', 'trials', 'melhor_trial', 'status']].round(2)

,modelo,versao_busca,mae_cv_motor,trials,melhor_trial,status,hiperparametros
0,Extra Trees,v1,25.01,100,77,COMPLETE,feature_selection=passthrough | n_estimators=4...
1,XGBoost,v1,25.30,100,1,COMPLETE,feature_selection=passthrough | n_estimators=2...
2,XGBoost,v2,25.32,100,76,COMPLETE,feature_selection=passthrough | n_estimators=1...
3,CatBoost,v1,25.35,100,2,COMPLETE,feature_selection=passthrough | iterations=316...
4,HistGradientBoosting,v2,25.43,100,14,COMPLETE,feature_selection=select_k_best | k_features=1...
5,HistGradientBoosting,v1,25.49,100,16,COMPLETE,feature_selection=select_k_best | k_features=1...


### Hiperparâmetros completos dos melhores trials

Os hiperparâmetros continuam armazenados na coluna `hiperparametros`. Como strings longas são truncadas visualmente quando aparecem dentro de uma tabela larga, elas serão apresentadas abaixo em blocos separados para facilitar a conferência dos valores e dos extremos da busca.

In [6]:
for row in comparison[['modelo', 'versao_busca', 'hiperparametros']].itertuples(index=False):
    print(f'{row.modelo} — {row.versao_busca}')
    print(f'  {row.hiperparametros}')
    print()

Extra Trees — v1
  feature_selection=passthrough | n_estimators=446 | max_depth=12 | min_samples_split=2 | min_samples_leaf=3 | max_features=0.642458

XGBoost — v1
  feature_selection=passthrough | n_estimators=232 | max_depth=4 | learning_rate=0.0227938 | min_child_weight=8 | subsample=0.772778 | colsample_bytree=0.716492 | gamma=0.305926 | reg_alpha=1.30602e-07 | reg_lambda=0.470147

XGBoost — v2
  feature_selection=passthrough | n_estimators=196 | max_depth=5 | learning_rate=0.0188086 | min_child_weight=11 | subsample=0.836787 | colsample_bytree=0.940402 | gamma=0.457042 | reg_alpha=2.80751e-07 | reg_lambda=3.38323

CatBoost — v1
  feature_selection=passthrough | iterations=316 | depth=7 | learning_rate=0.0145901 | l2_leaf_reg=2.70105 | random_strength=0.732724 | bagging_temperature=0.91214

HistGradientBoosting — v2
  feature_selection=select_k_best | k_features=14 | max_iter=487 | learning_rate=0.0101195 | max_leaf_nodes=18 | min_samples_leaf=43 | l2_regularization=0.595266

HistG

## 3. Comparação visual

A figura apresenta os melhores erros encontrados nas duas versões. Ela não substitui a confirmação final em um conjunto de teste mantido à parte; serve para mostrar se a ampliação direcionada mudou a posição dos candidatos.

In [5]:
fig = px.bar(
    comparison,
    x='modelo',
    y='mae_cv_motor',
    color='versao_busca',
    barmode='group',
    labels={
        'modelo': 'Modelo',
        'mae_cv_motor': 'Melhor MAE na validação por motor (ciclos)',
        'versao_busca': 'Busca',
    },
    title=lang({'title': 'Comparação entre as buscas v1 e v2'})['title'],
    height=550,
)
fig.show()

### Insights

Se as barras permanecerem próximas, será considerado encerrado o ciclo de ajuste fino desta etapa. O modelo escolhido deverá então ser reavaliado em uma confirmação final, sem utilizar o resultado do conjunto de teste para escolher entre configurações. Essa separação mantém a avaliação honesta: a validação orienta a escolha, enquanto o teste mede o desempenho depois que a escolha foi encerrada.

## Referências

AKIBA, T. et al. Optuna: A Next-generation Hyperparameter Optimization Framework. In: Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining, 2019.

PEDREGOSA, F. et al. Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, v. 12, p. 2825–2830, 2011.